# Ocean emulator: deterministic UNet

A monthly emulator of ACCESS-OM2 (1°, JRA55-do flux-forced) column **heat content (OHC)** and
**freshwater content**. Given the last two months and this month's surface forcing, it predicts
the next month; chaining that step forward gives a free-running ocean.

This is the **deterministic** version: one prediction per month. The ensemble version, with the same
structure, is `Ocean_Emulator_UNet_CRPS.ipynb`.

| Ingredient | What it does |
|---|---|
| **UNet** | the network: maps (last 2 states, forcing) to the change in the state |
| **MSE loss** | pointwise squared error, area-weighted over the ocean: pulls the prediction towards the expected next month |
| **Spectral loss (× 2)** | the prediction's energy in each band of zonal wavenumber must match ACCESS-OM2's: counters MSE's smoothing and stops grid-scale noise from growing |
| **Hard global budget constraint** | after every step, the ocean-total heat and freshwater change is set exactly to what the surface fluxes put in |

**Training is in two stages:**
1. **Stage 1, warm-up**: rollouts lengthening 1 → 4 → 12 months, every month scored.
2. **Stage 2, pushforward**: keep training the same network; now it runs freely for 24 (later 48) months
   without gradients, then the next 12 are scored. The model learns to correct the states it drifts into itself.

These are the settings of the `deterministic` emulator in `runs/intercomparison_90ep_v3` (seed 42).
Training takes ~35 min (stage 1) + ~1.4 h (stage 2) on one H200. To use that trained emulator instead
of retraining, set `RUN_DIR` to `REPO / "runs" / "intercomparison_90ep_v3"` in the settings cell.

**Reading the code:** nothing is imported from `src/`. Cells headed `IDENTICAL CODE EXISTS IN src/...`
are verbatim copies of the tested code in `src/`, placed where they are first used; every other cell is
written for this notebook only.

## 1. Setup

In [ ]:
# Standard Python
import os
from pathlib import Path

# Let the CUDA allocator grow memory segments instead of fragmenting them.
# This only works if it is set before anything touches the GPU, hence here at the top.
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

# Scientific Python
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr

# PyTorch (the network, the losses) and Lightning (the training loop)
import lightning as L
import torch
import torch.nn as nn
import torch.nn.functional as F
from lightning.pytorch.callbacks import ModelCheckpoint
from lightning.pytorch.loggers import CSVLogger
from torch.utils.checkpoint import checkpoint
from torch.utils.data import DataLoader

# Nothing is imported from src/: the cells headed "IDENTICAL CODE EXISTS IN src/..." are copies of it.
REPO = Path("/home/156/txs156/uom/OM2-emulator")

# Train on the GPU if there is one.
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"torch {torch.__version__} | device {DEVICE}")

## 2. Settings

Every setting is here.

In [ ]:
# ===================================== Data =========================================
DATA_PATH = "/g/data/nm47/txs156/OM2-emulator/data/1deg_ocean_heat_emulator_data_v3.nc"

# State variables: predicted by the model, then fed back in as its input for the next month.
PROGNOSTIC = ["ocean_heat_content_2d", "ocean_freshwater_content_2d"]

# Forcing variables: taken from ACCESS-OM2 every month, never predicted.
FORCING = ["total_surface_heat_flx", "ocean_freshwater_flux", "tau_x", "tau_y"]

# ===================================== Dates ========================================
# All dates are "YYYY-MM"; all ranges include both ends.
TIME_AXIS = ("1970-01", "2018-12")      # months loaded from the file
NORM_FIT = ("1970-01", "2000-12")       # months the climatology (mean, std) is computed from
TRAIN = ("1970-02", "2000-12")          # training samples start here; their whole window must end by 2000-12
VALID = ("2001-01", "2001-12")          # validation samples start in these months
TEST_START = "2005-02"                  # skill test: start from the true state of this month ...
TEST_END = "2018-11"                    # ... and predict up to one month after this (2018-12)
CONTROL_YEAR = ("2003-01", "2003-12")   # stability test: this year's forcing, repeated ...
CONTROL_YEARS = 50                      # ... this many times

# ==================================== Rollouts ======================================
N_PRIOR = 2         # the model sees 2 months (t-1 and t). The code below assumes 2.
WINDOW = 60         # months of truth every training sample needs after its start (48 free + 12 scored)
VALID_STEPS = 36    # validation: run 36 months from the true state and score all of them

# ===================================== Model ========================================
WIDTH_MULTIPLIER = 4   # UNet channels: 4 x 8 inputs = 32 at 1/2 resolution, 64 at 1/4, 128 in the bottleneck
SPECTRAL_WEIGHT = 2.0  # weight of the spectral loss (the main loss has weight 1)
# Spectral loss. Zonal spectra use only the latitude rows within SPECTRUM_LATITUDES: north of ~65N the
# ACCESS-OM2 tripole grid's nominal longitude is not a true longitude, so a row there is not a circle of
# latitude and its FFT has no clean physical wavenumber.
SPECTRUM_LATITUDES = (-65.0, 60.0)
SPECTRAL_BANDS_PER_DECADE = 8          # wavenumber bands per factor of 10 in zonal wavenumber

# ==================================== Training ======================================
# A rollout schedule says how long the training rollouts are, by epoch:
#   {first epoch: (free months without gradient, scored months)}
# e.g. {0: (0, 1), 15: (0, 4)} = epochs 0-14 score 1 month, from epoch 15 on score 4 months.
STAGE1_SCHEDULE = {0: (0, 1), 15: (0, 4), 45: (0, 12)}
STAGE1_EPOCHS = 90
STAGE1_LR = 1e-4        # learning rate (Adam), decayed to 0 over the stage by a cosine schedule
STAGE1_BATCH = 16       # samples per batch

STAGE2_SCHEDULE = {0: (24, 12), 45: (48, 12)}   # run 24 months freely and score the next 12; from epoch 45, 48 then 12
STAGE2_EPOCHS = 90
STAGE2_LR = 5e-5
STAGE2_BATCH = 16

# Memory saver for long scored rollouts: the network's intermediate values are kept only for the
# last 2 scored months; for the others they are recomputed during the backward pass (same result).
STEPS_IN_MEMORY = 2

SEED = 42   # random seed: initial weights, order of the training samples, noise

# ===================================== Outputs ======================================
RUN_DIR = REPO / "runs" / "unet_deterministic"       # trained weights and training logs, one folder per stage
FIG_DIR = REPO / "figures" / "unet_deterministic"    # figures
FIG_DIR.mkdir(parents=True, exist_ok=True)

# The normalised data, built once (a few minutes) and shared by both UNet notebooks.
FIELDS_FILE = Path(DATA_PATH).parent / "emulator_cache" / "unet_notebooks_fields_v3.pt"

## 3. Data

Every variable is turned into a **z-score**: `z = (x - mean) / std`, where mean and std are that grid
cell's value for that calendar month over 1970–2000 (the "spatial climatology"). So z = 0 is the
climatology and z = 1 a one-standard-deviation anomaly. Land is set to 0.

The whole record is normalised once and kept in memory as tensors with shape `(time, variable, lat, lon)`.
A training sample is then just its start month: its inputs and targets are slices of these tensors.

We also keep, for the budget constraint and for converting predictions back to physical units:
the climatological mean and std, the cell areas, the month lengths, and the ocean totals of every
forcing flux and of the climatological content.

The first cell below is the copied code that computes the climatology; the second uses it.

In [ ]:
# ==================================================================================================
# IDENTICAL CODE EXISTS IN src/Experiment/data.py
#   compute_normalisation
# Copied verbatim so the notebook shows everything it runs. Editing it here does NOT change src/
# (which the test suite checks), and vice versa.
# ==================================================================================================

def compute_normalisation(ds, variables, fit_period, mask_variable):
    """
    The monthly climatology of every variable: the mean and std of each grid
    cell for each calendar month, fitted over fit_period (first and last month,
    inclusive). Both are returned on ds's whole time axis: every month gets its
    calendar month's values. Also returns the ocean mask: 1 where
    mask_variable has data in the first month, 0 on land.

    ds has dimensions (time, latitude, longitude). The numbers are the same as
    the "Spatial_climatology" strategy of Data.build_normalisation (the
    PyEarthTools version used by the older notebooks); the test suite checks
    this.
    """
    mask = xr.where(np.isnan(ds[mask_variable].isel(time=0)), 0, 1)
    fit = ds[variables].where(mask == 1).sel(time=slice(*fit_period))

    # Mean and std of every cell for each calendar month, (month, latitude, longitude). The std is that of
    # the anomalies from the monthly mean.
    monthly_mean = fit.groupby("time.month").mean("time")
    monthly_std = (fit.groupby("time.month") - monthly_mean).groupby("time.month").std("time")

    # Give every month of ds its calendar month's values: (time, latitude, longitude).
    calendar_month = ds.time.dt.month
    mean = monthly_mean.sel(month=calendar_month).drop_vars("month")
    std = monthly_std.sel(month=calendar_month).drop_vars("month")
    return mean, std, mask

In [ ]:
if FIELDS_FILE.exists():
    # Built before: just load it.
    print(f"Loading the normalised data from {FIELDS_FILE}")
    saved = torch.load(FIELDS_FILE, weights_only=False)

else:
    print("Normalising the data (one pass over the file, a few minutes)...")

    # Open the file, rename the grid dimensions, and keep one value per month of TIME_AXIS.
    ds = xr.open_dataset(DATA_PATH).rename({"yt_ocean": "latitude", "xt_ocean": "longitude"})
    all_months = pd.date_range(TIME_AXIS[0], TIME_AXIS[1], freq="MS")
    ds = ds.sel(time=all_months, method="nearest")     # the file's time stamps are mid-month

    # The climatology: mean and std of every cell for every calendar month, fitted over NORM_FIT.
    # mean[v] and std[v] come back on the full time axis (each month gets its calendar month's values).
    # ocean_mask is 1 over ocean and 0 over land.
    mean, std, ocean_mask = compute_normalisation(ds, PROGNOSTIC + FORCING, NORM_FIT, mask_variable="total_surface_heat_flx")

    # z-scores of every variable, (time, lat, lon) each. Land is NaN in the file: set to 0.
    z = ((ds[PROGNOSTIC + FORCING] - mean) / std).fillna(0.0)

    # Ocean mask and cell area (m^2, set to 0 on land so every sum over cells is a sum over the ocean).
    mask_array = ocean_mask.values.astype(np.float32)
    area_array = np.nan_to_num(ds["area_t"].isel(time=0).values.astype(np.float64)) * mask_array

    # Everything as tensors. np.stack(..., axis=1) puts the variables side by side: (time, variable, lat, lon).
    saved = {
        "state_z": torch.tensor(np.stack([z[v].values for v in PROGNOSTIC], axis=1), dtype=torch.float32),
        "forcing_z": torch.tensor(np.stack([z[v].values for v in FORCING], axis=1), dtype=torch.float32),
        # Climatological mean and std of the state variables, physical units.
        "state_mean": torch.tensor(np.stack([mean[v].fillna(0.0).values for v in PROGNOSTIC], axis=1), dtype=torch.float32),
        "state_std": torch.tensor(np.stack([std[v].fillna(0.0).values for v in PROGNOSTIC], axis=1), dtype=torch.float32),
        "mask": torch.tensor(mask_array),
        "area": torch.tensor(area_array, dtype=torch.float64),
        # Ocean total of every forcing flux in every month (flux x area, summed over cells), e.g. W for the
        # heat flux: (time, forcing). Needed by the budget constraint.
        "forcing_total": torch.tensor(
            np.stack([(ds[v].fillna(0.0).values * area_array).sum(axis=(1, 2)) for v in FORCING], axis=1),
            dtype=torch.float64,
        ),
        # Length of every month in seconds (the budget constraint uses true month lengths).
        "month_seconds": torch.tensor([pd.Timestamp(t).days_in_month * 24 * 3600.0 for t in ds.time.values], dtype=torch.float64),
        "months": [pd.Timestamp(t).strftime("%Y-%m") for t in ds.time.values],
        "latitude": ds["latitude"].values,
        "longitude": ds["longitude"].values,
        "units": [ds[variable].attrs.get("units", "") for variable in PROGNOSTIC],
    }
    # Ocean total of the climatological content in every month, (time, state variable), e.g. J for OHC.
    saved["mean_total"] = (saved["state_mean"].double() * saved["area"]).sum(dim=(2, 3))

    FIELDS_FILE.parent.mkdir(parents=True, exist_ok=True)
    torch.save(saved, FIELDS_FILE)
    print(f"Saved {FIELDS_FILE}")

# Give everything a plain name.
state_z = saved["state_z"]              # (time, 2, lat, lon)  z-scores of OHC and freshwater content
forcing_z = saved["forcing_z"]          # (time, 4, lat, lon)  z-scores of the 4 forcings
state_mean = saved["state_mean"]        # (time, 2, lat, lon)  climatological mean, physical units
state_std = saved["state_std"]          # (time, 2, lat, lon)  climatological std, physical units
mask = saved["mask"]                    # (lat, lon)           1 = ocean, 0 = land
area = saved["area"]                    # (lat, lon)           cell area in m^2, 0 on land
months = saved["months"]                # ["1970-01", "1970-02", ...]
latitude = saved["latitude"]            # (lat,) nominal latitude of every grid row
longitude = saved["longitude"]          # (lon,)
units = saved["units"]                  # physical units of the 2 state variables

# Look up a month's position on the time axis: month_index["2005-02"] -> its time index.
month_index = {month: index for index, month in enumerate(months)}

# Training samples: every start month in TRAIN whose WINDOW following months also lie inside TRAIN,
# so training never sees a validation or test month.
train_start_months = torch.arange(month_index[TRAIN[0]], month_index[TRAIN[1]] + 1)
train_start_months = train_start_months[train_start_months + WINDOW <= month_index[TRAIN[1]]]

# Validation samples: every start month in VALID.
valid_start_months = torch.arange(month_index[VALID[0]], month_index[VALID[1]] + 1)

n_time, n_state, n_lat, n_lon = state_z.shape
print(f"{months[0]} .. {months[-1]}: {n_time} months on a {n_lat} x {n_lon} grid")
print(f"state  : {PROGNOSTIC}")
print(f"forcing: {FORCING}")
print(f"training samples: {len(train_start_months)} (start months {months[train_start_months[0]]} .. {months[train_start_months[-1]]})")
print(f"validation samples: {len(valid_start_months)}")

## 4. The hard global budget constraint

For column heat and freshwater content, the only way the **ocean total** can change is through the
surface. So between month a and month b:

$$\sum_\text{ocean} A\,(C_b - C_a) \;=\; \tfrac12\left(F_a L_a + F_b L_b\right)$$

where $C$ is the content per m², $A$ the cell area, $F$ the ocean-total surface flux and $L$ the month length.

After every prediction, `GlobalBudgetConstraint` computes how far the prediction is from this
(the residual $R$) and removes it by shifting the predicted z-score by the same amount at every ocean
cell. In physical units the correction is proportional to each cell's std, so it goes where the
content normally varies. The network still decides **where** heat and freshwater go; the constraint fixes
**how much** there is. Global drift is therefore impossible, for any run length.

The first cell below is the copied constraint; the second builds it and checks it on a "nothing changes" forecast.

In [ ]:
# ==================================================================================================
# IDENTICAL CODE EXISTS IN src/Experiment/constraints.py
#   GlobalBudgetConstraint
# Copied verbatim so the notebook shows everything it runs. Editing it here does NOT change src/
# (which the test suite checks), and vice versa.
# ==================================================================================================

class GlobalBudgetConstraint(nn.Module):
    """
    forward(pred, previous, previous_index, target_index) -> pred, with every
    enforced budget closed exactly.

    pred, previous   : (B, P, H, W) z-scores: the predicted next state and the
                       state it was predicted from
    previous_index,
    target_index     : (B,) time indices of those two states (their months)

    For each budget, with content C = z * std + mean (physical, per cell):

        R = sum(area * (C_pred - C_previous)) - sign * (G_a L_a + G_b L_b) / 2

    where G is the ocean-integrated surface flux of a month. The correction
    adds k * std_target to C_pred with k = -R / sum(area * std_target): a
    uniform shift of the z-score over the ocean, so it is spread where the
    content varies and never divides by a small std. It is linear in pred, so
    gradients pass through it.
    """

    def __init__(self, budgets, content_std, mean_global, flux_global, month_seconds, area, mask):
        """
        budgets      : [(name, content_channel, flux_channel, flux_sign)]
        content_std  : {name: (T, H, W) std of the content, physical units}
        mean_global  : (T, P) ocean integral of the climatological mean of each prognostic variable
        flux_global  : (T, C) ocean integral of each forcing variable (physical)
        month_seconds: (T,) month lengths in seconds
        area, mask   : (H, W) cell area (m^2) and ocean mask
        """
        super().__init__()
        self.budgets = [(name, int(c), int(f), float(sign)) for name, c, f, sign in budgets]
        ocean_area = (area.double() * mask.double())
        # Non-persistent: rebuilt from the data, so they stay out of checkpoints.
        self.register_buffer("ocean_area", ocean_area, persistent=False)
        self.register_buffer("ocean", (mask > 0).float(), persistent=False)
        self.register_buffer("mean_global", mean_global.double(), persistent=False)
        self.register_buffer("flux_global", flux_global.double(), persistent=False)
        self.register_buffer("month_seconds", month_seconds.double(), persistent=False)
        for name, *_ in self.budgets:
            self.register_buffer(f"{name}_std", content_std[name].float(), persistent=False)

    def residual(self, pred, previous, previous_index, target_index, budget):
        """(B,) budget residual R (content units x m^2) of one budget, and sum(area * std_target)."""
        name, c, f, sign = budget
        std = getattr(self, f"{name}_std")
        std_previous = std[previous_index].double()
        std_target = std[target_index].double()
        area = self.ocean_area
        change = ((pred[:, c].double() * std_target - previous[:, c].double() * std_previous) * area).sum(dim=(-2, -1))
        change = change + self.mean_global[target_index, c] - self.mean_global[previous_index, c]
        flux = sign * 0.5 * (self.flux_global[previous_index, f] * self.month_seconds[previous_index]
                             + self.flux_global[target_index, f] * self.month_seconds[target_index])
        return change - flux, (std_target * area).sum(dim=(-2, -1))

    def forward(self, pred, previous, previous_index, target_index):
        out = pred.clone()
        for budget in self.budgets:
            residual, weight = self.residual(out, previous, previous_index, target_index, budget)
            k = (residual / weight.clamp_min(1e-30)).to(pred.dtype)
            out[:, budget[1]] = out[:, budget[1]] - k[:, None, None] * self.ocean
        return out

In [ ]:
# Channel positions: which state channel is which content, which forcing channel is which flux.
ohc_channel = PROGNOSTIC.index("ocean_heat_content_2d")              # 0
freshwater_channel = PROGNOSTIC.index("ocean_freshwater_content_2d")  # 1
heat_flux_channel = FORCING.index("total_surface_heat_flx")           # 0
freshwater_flux_channel = FORCING.index("ocean_freshwater_flux")      # 1

constraint = GlobalBudgetConstraint(
    # One entry per budget: (name, content channel, flux channel, sign). Sign +1: positive flux = into the ocean.
    budgets=[
        ("heat", ohc_channel, heat_flux_channel, 1.0),
        ("freshwater", freshwater_channel, freshwater_flux_channel, 1.0),
    ],
    content_std={"heat": state_std[:, ohc_channel], "freshwater": state_std[:, freshwater_channel]},
    mean_global=saved["mean_total"],         # ocean total of the climatological content, every month
    flux_global=saved["forcing_total"],      # ocean total of every forcing flux, every month
    month_seconds=saved["month_seconds"],    # month lengths
    area=area,
    mask=mask,
)

# Check: forecast March 2005 as "nothing changes" (same z-scores as February), then apply the constraint.
# The residual is (change in ocean-total content) - (what the surface flux put in), so it should be ~0 after.
february = month_index["2005-02"]
previous_state = state_z[february : february + 1]          # (1, 2, lat, lon), the true February 2005
forecast = previous_state.clone()                          # "nothing changes"
previous_month = torch.tensor([february])
target_month = torch.tensor([february + 1])
constrained_forecast = constraint(forecast, previous_state, previous_month, target_month)

for budget in constraint.budgets:
    residual_before, _ = constraint.residual(forecast, previous_state, previous_month, target_month, budget)
    residual_after, _ = constraint.residual(constrained_forecast, previous_state, previous_month, target_month, budget)
    print(f"{budget[0]:>10} budget residual: {residual_before.item():+.3e} before, {residual_after.item():+.3e} after")

## 5. The model

`ForwardEmulator` wraps the `UNet`:

    next state = current state + UNet(state at t-1, state at t, forcing at t+1)

- The UNet predicts the **change**, so "nothing changes" costs it nothing and it only learns the tendency.
- Inputs: 2 months × 2 state variables + 4 forcings = 8 channels.
- Inside the UNet: two stride-2 downsamplings (to 1/4 resolution), and **partial convolutions**
  (Liu et al. 2018): each convolution sees only the ocean cells in its window (land is multiplied by 0) and
  its output is scaled up by (window size) / (number of ocean cells in it), so coastal cells are not damped.
  Before every convolution the field is padded to follow the grid: it wraps around in longitude and
  follows the tripole fold at the north edge.

The two copied cells below hold the UNet (with its partial convolutions), then `ForwardEmulator`
(adapted from `src/`: the noise inputs it has there, for the CRPS ensemble, are removed). The third builds the network.

In [ ]:
# ==================================================================================================
# IDENTICAL CODE EXISTS IN src/Emulator/om2_model_utils.py
#   PartialConv2d, upsample_periodic_x, UNet
# Copied verbatim so the notebook shows everything it runs. Editing it here does NOT change src/
# (which the test suite checks), and vice versa.
# ==================================================================================================

class PartialConv2d(nn.Module):
    """
    A convolution that sees only ocean cells: a partial convolution (Liu et al. 2018,
    "Image Inpainting for Irregular Holes Using Partial Convolutions").

    forward(x, mask) -> (out, new_mask)
        x        : (B, C_in, H, W)
        mask     : (B, 1, H, W), or (1, 1, H, W) shared by the whole batch; 1 = ocean, 0 = land
        out      : (B, C_out, H_out, W_out)
        new_mask : 1 wherever the convolution window held at least one ocean cell

    Three steps:
    1. Pad x and the mask with ``padding`` cells on every side, following the ACCESS-OM2 tripolar grid:
         * east/west: longitude is periodic, so the columns wrap around. (With non-periodic padding the
           seam acted as a coastline neither side could see across; in the autoregressive OHC emulator
           98% of the fastest-growing perturbation sat at that seam.)
         * north: the tripole fold. Top-row cell i neighbours top-row cell nx - 1 - i, so the ghost rows
           above the top row are the top rows reversed in x and in y.
         * south (Antarctica): rows of zeros, marked as land, so the edge acts as a coastline.
    2. Convolve x * mask, i.e. with land set to 0.
    3. Multiply by (cells in the window) / (ocean cells in the window), so values next to a coast are
       not damped by the land around them. 0 where the window holds no ocean cell.
    """

    def __init__(self, in_ch, out_ch, kernel_size=3, stride=1, padding=1):
        super().__init__()
        self.padding = padding
        self.conv = nn.Conv2d(in_ch, out_ch, kernel_size=kernel_size, stride=stride)
        # Counts the ocean cells in every window: all-ones weights, never trained.
        self.mask_conv = nn.Conv2d(1, 1, kernel_size=kernel_size, stride=stride, bias=False)
        self.mask_conv.weight.data[:] = 1.0
        self.mask_conv.requires_grad_(False)
        self.kernel_area = kernel_size * kernel_size

    def forward(self, x, mask):
        p = self.padding
        # 1. Padding. North: the top p rows flipped in y and x (rows run south to north).
        x_fold = torch.flip(x[..., -p:, :], dims=(-2, -1))
        mask_fold = torch.flip(mask[..., -p:, :], dims=(-2, -1))
        # Stack [p zero rows (south), field, fold rows (north)] along y, then wrap p columns around in x.
        x = F.pad(torch.cat([F.pad(x, (0, 0, p, 0)), x_fold], dim=-2), (p, p, 0, 0), mode="circular")
        mask = F.pad(torch.cat([F.pad(mask, (0, 0, p, 0)), mask_fold], dim=-2), (p, p, 0, 0), mode="circular")

        # 2 and 3. Convolve the ocean cells only, and scale up by (cells / ocean cells) in each window.
        ocean_cells = self.mask_conv(mask)
        scale = torch.where(ocean_cells > 0, self.kernel_area / (ocean_cells + 1e-6), torch.zeros_like(ocean_cells))
        new_mask = (ocean_cells > 0).to(mask.dtype)
        return self.conv(x * mask) * scale, new_mask


def upsample_periodic_x(x, size):
    """
    Bilinear upsampling of x (B, C, h, w) to size = (H, W), with longitude (the last dimension)
    periodic: wrap one column around on each side, interpolate, then cut the extra columns off again,
    so the east and west edges interpolate across the seam. W must be a whole multiple of w.
    """
    scale = size[1] // x.shape[-1]          # e.g. 2: every input column becomes 2 output columns
    x = F.pad(x, (1, 1, 0, 0), mode="circular")
    x = F.interpolate(x, size=(size[0], size[1] + 2 * scale), mode="bilinear", align_corners=False)
    return x[..., scale:-scale]


class UNet(nn.Module):
    """
    Mask-aware U-Net: two stride-2 encoder levels, a bottleneck, and a decoder with skip connections at
    every resolution. Every convolution is a PartialConv2d, so land never enters the computation.

    forward(x, mask) -> (B, output_channel_count, H, W)
        x    : (B, input_channel_count, H, W) inputs (prior states, forcing, [noise])
        mask : (1, 1, H, W) or (B, 1, H, W) ocean mask, 1 = ocean

    Channel widths scale with the number of inputs, so adding predictors widens the network instead of
    squeezing them through a fixed bottleneck:
        level 1 (1/2 resolution)   : width_multiplier * width_channel_count   (32 for 8 inputs and 4)
        level 2 (1/4 resolution)   : 2 x level 1
        bottleneck (1/4 resolution): 4 x level 1
    width_channel_count defaults to input_channel_count. Setting it lets extra inputs (the noise
    channels of the CRPS ensemble) leave every layer but the first unchanged, so a trained network's
    weights still fit. width_multiplier must be >= 2 for the first layer to carry every signed input
    through its ReLU (each needs a +/- pair of channels).

    Design notes:
      * The bottleneck (enc3a-c) and the first decoder layer (dec1a-c) are each three 3x3 convolutions
        with a ReLU after each: together a 7x7 window (3 -> 5 -> 7), with 27 instead of 49 weights per
        channel pair and two extra non-linearities.
      * The output layer sees only the upsampled decoder features, not the raw input: fed the raw input
        as well, it learned a sharpening stencil that grew grid-scale noise by ~17% per step. The
        residual wrapper (next = state + output) carries the state's grid-scale content forward instead.
    """

    def __init__(self,
                 input_channel_count=2,
                 output_channel_count=2,
                 width_multiplier=4,
                 width_channel_count=None):
        super().__init__()

        # Channel widths, scaled from the number of (physical) input channels.
        width1 = int(round(width_multiplier * (width_channel_count or input_channel_count)))  # 1/2 resolution
        width2 = 2 * width1                                                                     # 1/4 resolution
        latent_width = 4 * width1                                                               # bottleneck

        # ---------- Encoder ----------
        self.enc1 = PartialConv2d(input_channel_count, width1, kernel_size=4, stride=2, padding=1)
        self.enc2 = PartialConv2d(width1, width2, kernel_size=3, stride=2, padding=1)
        # Bottleneck: three 3x3 layers, the receptive field of one 7x7.
        self.enc3a = PartialConv2d(width2, width2, kernel_size=3, stride=1, padding=1)
        self.enc3b = PartialConv2d(width2, width2, kernel_size=3, stride=1, padding=1)
        self.enc3c = PartialConv2d(width2, latent_width, kernel_size=3, stride=1, padding=1)

        # ---------- Decoder ----------
        # After the first upsample: latent + skip from enc2 (three 3x3 layers, as a 7x7).
        self.dec1a = PartialConv2d(latent_width + width2, width2, kernel_size=3, stride=1, padding=1)
        self.dec1b = PartialConv2d(width2, width2, kernel_size=3, stride=1, padding=1)
        self.dec1c = PartialConv2d(width2, width2, kernel_size=3, stride=1, padding=1)

        # After the second upsample: dec1 + skip from enc1.
        self.dec2 = PartialConv2d(width2 + width1, width1, kernel_size=3, stride=1, padding=1)

        # Full resolution: dec2 upsampled.
        self.dec3 = PartialConv2d(width1, output_channel_count, kernel_size=3, stride=1, padding=1)

        self.relu = nn.ReLU(inplace=True)

    def forward(self, x, mask):
        # The whole network on one batch. Every self.encN / self.decN(...) call runs one PartialConv2d
        # with that layer's own weights and returns (features, mask). Example widths: 8 inputs, width_multiplier 4.
        # ================================ Encoder ================================
        # PartialConv2d.forward, 4x4 stride 2: full -> 1/2 resolution, C_in -> 32 channels.
        x1, mask1 = self.enc1(x, mask)
        x1 = self.relu(x1)

        # PartialConv2d.forward, 3x3 stride 2: 1/2 -> 1/4 resolution, 32 -> 64 channels.
        x2, mask2 = self.enc2(x1, mask1)
        x2 = self.relu(x2)

        # Bottleneck, at 1/4 resolution: three 3x3 PartialConv2d.forward calls, a ReLU after each.
        latent, latent_mask = self.enc3a(x2, mask2)                  # 64 -> 64 channels
        latent = self.relu(latent)
        latent, latent_mask = self.enc3b(latent, latent_mask)        # 64 -> 64
        latent = self.relu(latent)
        latent, latent_mask = self.enc3c(latent, latent_mask)        # 64 -> 128
        latent = self.relu(latent)

        # ================================ Decoder ================================
        # x1 and x2 (the encoder features at 1/2 and 1/4 resolution) are the skip
        # connections: they are joined back in at their own resolution.

        # The bottleneck is already at x2's 1/4 resolution: upsample_periodic_x
        # interpolates to x2's grid size, wrapping around in longitude.
        y = upsample_periodic_x(latent, x2.shape[2:])
        y_mask = F.interpolate(latent_mask, size=mask2.shape[2:], mode="nearest")
        y = torch.cat([y, x2], dim=1)                                # skip from enc2: 128 + 64 = 192 channels

        # Three 3x3 PartialConv2d.forward calls at 1/4 resolution, a ReLU after each.
        y, y_mask = self.dec1a(y, y_mask)                            # 192 -> 64
        y = self.relu(y)
        y, y_mask = self.dec1b(y, y_mask)                            # 64 -> 64
        y = self.relu(y)
        y, y_mask = self.dec1c(y, y_mask)                            # 64 -> 64
        y = self.relu(y)

        # Upsample 1/4 -> 1/2 resolution and join the skip from enc1.
        y = upsample_periodic_x(y, x1.shape[2:])
        y_mask = F.interpolate(y_mask, size=mask1.shape[2:], mode="nearest")
        y = torch.cat([y, x1], dim=1)                                # 64 + 32 = 96 channels

        # PartialConv2d.forward, 3x3 at 1/2 resolution.
        y, y_mask = self.dec2(y, y_mask)                             # 96 -> 32
        y = self.relu(y)

        # Upsample 1/2 -> full resolution (the input's exact size, odd grids included).
        y = upsample_periodic_x(y, x.shape[2:])

        # PartialConv2d.forward, 3x3 at full resolution with the original ocean mask:
        # the output, one channel per predicted variable. No ReLU: the change can be negative.
        change, _ = self.dec3(y, mask)                               # 32 -> C_out
        return change

In [ ]:
# ==================================================================================================
# ADAPTED FROM src/Experiment/models.py
#   ForwardEmulator
# Copied, then the noise inputs (noise_channels, used only by the CRPS ensemble) removed; otherwise identical.
# Editing it here does NOT change src/ (which the test suite checks), and vice versa.
# ==================================================================================================

class ForwardEmulator(nn.Module):
    """
    One-step forward emulator: next state = most recent state + tendency_network(inputs).

    forward(prior, forcing, mask) -> (B, P, H, W) the next state, in float32 (so the losses stay in full
    precision when training with 16-bit mixed precision)
        prior   : (B, n_prior * P, H, W) prior states stacked oldest first
        forcing : (B, C, H, W) forcing of the month being predicted
        mask    : (H, W) ocean mask, 1 = ocean

    The tendency network (the UNet, or the Samudra ConvNeXt U-Net) predicts the CHANGE from the most
    recent state, so persisting the current anomaly costs it nothing and it only has to learn the
    tendency. Land values are ignored: the losses and the evaluation mask them out.
    """

    def __init__(self, tendency_network, n_prognostic):
        super().__init__()
        self.tendency_network = tendency_network
        self.n_prognostic = n_prognostic          # P: the most recent state is the last P channels of prior

    def forward(self, prior, forcing, mask):
        # The inputs side by side, laid out channels-last in memory (NHWC): faster 16-bit convolutions on
        # tensor cores, same results.
        x = torch.cat([prior, forcing], dim=1).contiguous(memory_format=torch.channels_last)
        # One (1, 1, H, W) land mask broadcast over the batch: every partial convolution then counts the
        # ocean cells once, not once per sample.
        mask = mask.to(device=x.device, dtype=x.dtype)[None, None]
        next_state = prior[:, -self.n_prognostic :] + self.tendency_network(x, mask)
        return next_state.float()

In [ ]:
# Input channels: 2 months x 2 state variables + 4 forcings = 8.
n_inputs = N_PRIOR * len(PROGNOSTIC) + len(FORCING)

L.seed_everything(SEED)   # same initial weights every time

unet = UNet(
    input_channel_count=n_inputs,
    output_channel_count=len(PROGNOSTIC),   # it predicts the change in both state variables
    width_multiplier=WIDTH_MULTIPLIER,
)
model = ForwardEmulator(unet, n_prognostic=len(PROGNOSTIC))   # next state = current state + UNet(...)
model = model.to(memory_format=torch.channels_last)   # weights channels-last, like the inputs: faster 16-bit convolutions

n_parameters = sum(parameter.numel() for parameter in model.parameters())
print(f"UNet parameters: {n_parameters:,}")

## 6. Loss functions

Both work in z-score units.

**MSE**, per cell $(x - y)^2$, averaged over ocean cells **weighted by cell area** (so a polar cell counts
less than an equatorial one). Its best prediction is the *expected* next month: it captures the response
to the forcing but averages away anything it cannot predict, so long rollouts get smoother.

**Spectral** (weight 2): take every latitude row from 65°S to 60°N, Fourier transform it in longitude, and
add up the energy in log-spaced bands of physical zonal wavenumber (cycles/km). The loss is the mean
squared difference of log10(energy) between prediction and truth, over bands. It ignores *where*
features are (phase), only *how energetic* each scale is, so it penalises both blurring (too little
large-scale energy) and grid-scale noise (too much small-scale energy).

The two pull against each other: MSE penalises energy in the wrong place, the spectral term asks for
the right amount of energy wherever it is.

The first cell below is the copied loss code; the second picks the losses.

In [ ]:
# ==================================================================================================
# IDENTICAL CODE EXISTS IN src/Emulator/om2_loss_functions.py
#   field_loss, EARTH_RADIUS_KM, zonal_wavenumber_bands, spectral_loss
# (src also defines SPECTRUM_LATITUDES and SPECTRAL_BANDS_PER_DECADE here; in this notebook they are
# set in the settings cell at the top.)
# Copied verbatim so the notebook shows everything it runs. Editing it here does NOT change src/
# (which the test suite checks), and vice versa.
# ==================================================================================================

def field_loss(*, pred_t, target_t, mask, area, method="mse", n_members=1, **_):
    """
    Ocean-masked, AREA-WEIGHTED mean of a per-cell error for one rollout step,
    in z-score units. ``method`` chooses the per-cell error:

      "mse"   squared error (pred_t - target_t)^2.

      "crps"  fair CRPS of an ensemble. The batch holds ``n_members`` (from the
              rollout context) consecutive members of each sample: pred_t is
              (B * M, ...) and target_t repeats the truth for every member.
              Per cell, the fair (unbiased in M) CRPS is

                  mean_i |x_i - y|  -  sum_{i,j} |x_i - x_j| / (2 M (M - 1))

              (Ferro 2014; used for ensemble training by e.g. NeuralGCM and
              AIFS-CRPS). The first term pulls every member towards the truth,
              the second pushes members apart, so the loss is minimised by
              members drawn from the distribution of the truth, not by their
              mean: sharp members are not penalised for unpredictable detail as
              they are by MSE.

    The per-cell error is then averaged over the ocean, each cell weighted by
    its area (``area`` from the rollout context, m^2), like every evaluation
    metric, the per-variable RMSE logged in training, and the closure terms. On
    the ACCESS-OM2 1-degree grid cell areas vary 17x (latitudinal refinement
    near the equator, small cells near the poles): an unweighted per-cell mean
    gave the regions poleward of 60 degrees ~27% of the loss for ~10% of the
    ocean area.

    Each training setup chooses ``method`` once when it builds its list of
    losses (the loss cell of the UNet notebooks, LOSS_TERMS in
    src/Experiment/losses.py); rollout-context entries it does not use are
    absorbed by ``**_``. The loss weight is applied by the caller.
    """
    if method == "mse":
        error = (pred_t - target_t) ** 2                               # (B, P, H, W)
    elif method == "crps":
        m = int(n_members)
        if m < 2:
            raise ValueError("field_loss(method='crps') needs an ensemble: n_members >= 2 in the rollout context")
        b = pred_t.shape[0] // m
        members = pred_t.reshape(b, m, *pred_t.shape[1:])
        truth = target_t.reshape(b, m, *target_t.shape[1:])[:, 0]
        skill = (members - truth[:, None]).abs().mean(dim=1)
        spread = (members[:, :, None] - members[:, None]).abs().sum(dim=(1, 2)) / (m * (m - 1))
        error = skill - 0.5 * spread                                    # (B, P, H, W)
    else:
        raise ValueError(f"Unknown field_loss method {method!r}: use 'mse' or 'crps'")

    # Area-weighted mean over the ocean. Ocean mask (H, W), 1 = ocean, broadcast to error's (B, P, H, W).
    cell_weight = (mask * area).to(device=error.device, dtype=error.dtype).expand_as(error)
    return (error * cell_weight).sum() / cell_weight.sum().clamp_min(1e-12)


EARTH_RADIUS_KM = 6371.0


def zonal_wavenumber_bands(latitude, n_lon):
    """
    Sort every (latitude row, zonal harmonic) into bands of PHYSICAL zonal
    wavenumber k (cycles/km), so a band is the same length scale at every
    latitude and at every resolution (1 deg or 1/10 deg).

    Returns (rows, band, power_weight, k_band):
      rows          indices of the latitude rows used (those within SPECTRUM_LATITUDES);
      band          (rows * harmonics,) band number of each (row, harmonic),
                    flattened row by row;
      power_weight  (rows, harmonics): multiplies |rfft|^2 so that it sums to
                    the row's variance, times cos(latitude) (the row's area share);
      k_band        (n_bands,) central wavenumber of each band (cycles/km).
    """
    latitude = np.asarray(latitude, dtype=float)

    # Latitude rows used (north of ~65N the tripole grid's rows are not circles of latitude).
    rows = np.flatnonzero((latitude >= SPECTRUM_LATITUDES[0]) & (latitude <= SPECTRUM_LATITUDES[1]))
    cos_lat = np.cos(np.deg2rad(latitude[rows]))

    # Physical wavenumber of every (row, harmonic). A row is a circle of length 2 pi R cos(lat),
    # and harmonic n (n waves around it) has k = n / length. rfftfreq(n_lon, d=1/n_lon) lists the
    # harmonics rfft returns, 0 .. n_lon // 2; harmonic 0 is the zonal mean, which is dropped.
    row_length_km = 2 * np.pi * EARTH_RADIUS_KM * cos_lat              # (rows,)
    harmonics = np.fft.rfftfreq(n_lon, d=1.0 / n_lon)[1:]              # (harmonics,)
    k = harmonics[None, :] / row_length_km[:, None]                    # (rows, harmonics)

    # Bands: equal steps in log10(k), SPECTRAL_BANDS_PER_DECADE of them per factor of 10, counted from
    # the smallest k. Then renumber them 0, 1, 2, ..., skipping any band no (row, harmonic) falls in.
    log_k = np.log10(k)
    band = np.floor((log_k - log_k.min()) * SPECTRAL_BANDS_PER_DECADE).astype(int)
    used, band = np.unique(band.ravel(), return_inverse=True)
    k_band = 10.0 ** (log_k.min() + (used + 0.5) / SPECTRAL_BANDS_PER_DECADE)  # band centres

    # Parseval: sum_n |rfft_n|^2 / n_lon^2 is the row's variance, once every harmonic below
    # Nyquist is counted twice (rfft returns n but not its mirror image -n, which has the same
    # power). Times cos(lat), the row's share of the area.
    counted_twice = np.where(harmonics < n_lon / 2, 2.0, 1.0)
    power_weight = cos_lat[:, None] * counted_twice[None, :] / n_lon**2
    return rows, band, power_weight, k_band


def spectral_loss(*, latitude, n_lon):
    """
    Return the spectral loss for a grid with these row latitudes and n_lon
    longitudes: the mean squared difference in log10 ENERGY per physical
    zonal-wavenumber band, per sample and variable.

    For every latitude row in SPECTRUM_LATITUDES the ocean-masked field
    (land = 0) is Fourier transformed along longitude, which is periodic. The
    power of each harmonic is summed into log-spaced bands of physical zonal
    wavenumber (zonal_wavenumber_bands), weighting rows by cos(latitude) so the
    band energies add up to the area-weighted zonal variance. The loss compares
    log energies band by band.

    Why this form. The skill-test spectra showed two failures over long
    rollouts: large scales losing energy (the MSE fit damps anomalies, i.e.
    blurring) and small scales gaining it (grid-scale noise growing). A loss on
    band energies penalises both, in either direction. It does not ask for the
    right phase, so an unpredictable eddy in the wrong place is not penalised
    as if it were a wrong large-scale pattern, and it does not suppress small
    scales that are really there -- the point at 1/10 deg. Logs make every band
    count, not just the energetic large scales. (The previous version compared
    the amplitude of every individual 2-D Fourier mode: noisy, and its FFT ran
    across the non-periodic north/south edges.)

    ``latitude``: the grid's 1-D nominal latitude (degrees), which sets each
    row's length and so its physical wavenumbers. ``n_lon``: number of
    longitudes.

    The returned loss is called with the whole rollout context
    (``loss(**context)``); the loss weight is applied by the caller.
    """
    # The grid's band layout (see zonal_wavenumber_bands), computed once. The tensors are
    # moved to the field's device at every call: a few hundred kB, negligible next to the network.
    rows, band, power_weight, k_band = zonal_wavenumber_bands(latitude, n_lon)
    rows, band = torch.as_tensor(rows), torch.as_tensor(band)
    power_weight = torch.as_tensor(power_weight, dtype=torch.float32)

    def log_band_energy_mse(pred_t, target_t, mask):
        device = pred_t.device
        # Prediction and truth side by side with land set to 0, only the rows used: (2, B, P, rows, n_lon).
        # In float32: half-precision FFTs need power-of-two sizes.
        fields = torch.stack([pred_t.float(), target_t.float()]) * mask.to(device, torch.float32)
        fields = fields[..., rows.to(device), :]
        # Power of every zonal harmonic except the zonal mean, as its share of the area-weighted variance.
        power = torch.fft.rfft(fields, dim=-1).abs().square()[..., 1:] * power_weight.to(device)
        # Energy per wavenumber band: add up the power of every (row, harmonic) in it. (2, B, P, n_bands)
        energy = power.new_zeros(*power.shape[:-2], k_band.size).index_add(-1, band.to(device), power.flatten(-2))
        pred_energy, target_energy = energy
        # Floor relative to each sample's total energy, so a band with no ocean
        # (or a zero field) never gives log(0).
        floor = 1.0e-8 * target_energy.sum(-1, keepdim=True) + 1.0e-20
        return (torch.log10(pred_energy + floor) - torch.log10(target_energy + floor)).square().mean()

    def loss_fn(*, pred_t, target_t, mask, **_):
        # Gradient checkpointing: the FFTs are recomputed in the backward pass instead of stored
        # (same value and gradients, less memory). Without gradients this is a plain call.
        return checkpoint(log_band_energy_mse, pred_t, target_t, mask, use_reentrant=False)

    return loss_fn

In [ ]:
# Each loss is a function that the training loop calls at every scored month as
#   loss_function(pred_t=..., target_t=..., mask=..., area=..., n_members=...)
# and that returns one number. All of them take the same inputs, even the ones they do not use.


def mse(pred_t, target_t, mask, area, n_members):
    # Area-weighted mean squared error over the ocean (n_members is not used).
    return field_loss(pred_t=pred_t, target_t=target_t, mask=mask, area=area, method="mse")


# Log energy per zonal-wavenumber band. Built here, once, because it needs the grid:
# the row latitudes and the number of longitudes.
spectral = spectral_loss(latitude=latitude, n_lon=mask.shape[-1])

# The losses: {name: (weight, loss function)}. They are added up at every scored month, in both stages.
LOSSES = {"mse": (1.0, mse), "spectral": (SPECTRAL_WEIGHT, spectral)}

## 7. Rollout training

A training batch is a set of start months. For each one:

1. start from the **true** states of the start month and the month before;
2. run `n_free` months on the model's own predictions **without gradients** ("pushforward": cheap,
   and it puts the model in the states it actually drifts into);
3. run `n_scored` more months **with gradients**, scoring every month against ACCESS-OM2.

Every prediction passes through the budget constraint before it is scored and fed back.

Training uses 16-bit mixed precision, the Adam optimiser and a cosine learning-rate decay.
Validation always runs `VALID_STEPS` = 36 months from the true state, so its loss is comparable between epochs.

The copied cell below resumes an interrupted stage and reads its training log; the next one defines the
training itself.

In [ ]:
# ==================================================================================================
# IDENTICAL CODE EXISTS IN src/Experiment/training.py
#   prepare_resume, read_metrics
# Copied verbatim so the notebook shows everything it runs. Editing it here does NOT change src/
# (which the test suite checks), and vice versa.
# ==================================================================================================

def prepare_resume(run_dir):
    """
    The checkpoint to resume run_dir's training from, or None if there is none
    yet. ModelCheckpoint (see build_trainer) overwrites checkpoints/last.ckpt
    after every epoch, with the weights, optimizer, learning-rate schedule and
    epoch number, so trainer.fit(..., ckpt_path=prepare_resume(run_dir))
    carries on from the last completed epoch (the rollout schedule follows the
    epoch number). When resuming, metrics.csv is renamed to metrics_part<N>.csv
    first, because the CSV logger replaces metrics.csv when training starts;
    read_metrics joins the parts.
    """
    checkpoint = Path(run_dir, "checkpoints", "last.ckpt")
    if not checkpoint.exists():
        return None
    # A run stopped before its first log write has a checkpoint but no metrics.csv yet.
    log = Path(run_dir, "metrics.csv")
    if log.exists():
        n_parts = len(list(Path(run_dir).glob("metrics_part*.csv")))
        log.rename(log.with_name(f"metrics_part{n_parts}.csv"))
    return checkpoint


def read_metrics(run_dir):
    """
    The CSV training log of run_dir as one row per epoch: metrics_part0.csv, metrics_part1.csv, ...
    (from earlier, interrupted runs; see prepare_resume) then metrics.csv, keeping the latest value of
    every column for each epoch.
    """
    run_dir = Path(run_dir)
    # The parts in number order (as text, part10 would sort before part2), then the current log.
    parts = sorted(run_dir.glob("metrics_part*.csv"), key=lambda path: int(path.stem[len("metrics_part"):]))
    frames = [pd.read_csv(path) for path in [*parts, run_dir / "metrics.csv"] if path.exists()]
    return pd.concat(frames, ignore_index=True).groupby("epoch", as_index=False).last()

In [ ]:
class RolloutTraining(L.LightningModule):
    # Lightning runs the training loop for us. Every epoch it calls training_step on every batch of
    # training start months and validation_step on every batch of validation start months, and it uses
    # configure_optimizers to update the weights. A batch is a 1-D tensor of start months (time indices).

    def __init__(self, model, losses, schedule, lr, epochs, members):
        super().__init__()
        self.model = model             # the ForwardEmulator being trained
        self.losses = losses           # {name: (weight, loss function)}
        self.schedule = schedule       # {first epoch: (free months, scored months)}
        self.lr = lr                   # learning rate
        self.epochs = epochs           # length of the stage (for the cosine learning-rate decay)
        self.members = members         # rollouts per sample: 1 = deterministic, 4 = CRPS ensemble
        self.constraint = constraint   # the budget constraint from section 4

        # Copies of the data held by this module, so that Lightning moves them to the GPU with it and a
        # batch can be cut out of them on the GPU. persistent=False: not written into checkpoints.
        self.register_buffer("state_z", state_z, persistent=False)
        self.register_buffer("forcing_z", forcing_z, persistent=False)
        self.register_buffer("mask", mask, persistent=False)
        self.register_buffer("area", area.float(), persistent=False)

    def rollout_loss(self, start_months, n_free, n_scored):
        # Roll out every sample from its start month. Returns the loss averaged over the scored months,
        # and each loss term separately (for the training log).

        # Each sample is rolled out `members` times (different noise each time), the copies side by side:
        # start months [a, b] with 4 members -> [a, a, a, a, b, b, b, b]. With 1 member nothing changes.
        t = start_months.repeat_interleave(self.members)

        # The model's input state: the true months t-1 and t, stacked as channels
        #   [OHC(t-1), freshwater(t-1), OHC(t), freshwater(t)]   -> shape (batch, 4, lat, lon)
        state = torch.cat([self.state_z[t - 1], self.state_z[t]], dim=1)

        # ---- Free months: the model runs on its own predictions, with no gradients ----
        with torch.no_grad():
            for month in range(n_free):
                prediction = self.model(state, self.forcing_z[t + 1], self.mask)          # month t+1, (batch, 2, lat, lon)
                prediction = self.constraint(prediction, state[:, n_state:], t, t + 1)    # close the global budgets
                state = torch.cat([state[:, n_state:], prediction], dim=1)                # drop month t-1, add month t+1
                t = t + 1

        # Under mixed precision PyTorch caches 16-bit copies of the weights. Copies made just now, without
        # gradients, would cut the scored months off from the weights, so throw them away.
        torch.clear_autocast_cache()

        # ---- Scored months: same steps, now with gradients, and every month compared with the truth ----
        # Each loss term, averaged over the scored months.
        loss_parts = {name: 0.0 for name in self.losses}
        for month in range(n_scored):
            # Memory saver for all but the last STEPS_IN_MEMORY months: don't keep the network's intermediate
            # values, recompute them in the backward pass instead (same result, ~30% more compute).
            # Without gradients (validation) checkpoint is just a plain call.
            if month < n_scored - STEPS_IN_MEMORY:
                prediction = checkpoint(self.model, state, self.forcing_z[t + 1], self.mask, use_reentrant=False)
            else:
                prediction = self.model(state, self.forcing_z[t + 1], self.mask)
            prediction = self.constraint(prediction, state[:, n_state:], t, t + 1)

            # Score the prediction against ACCESS-OM2 in that month, with every loss term.
            for name, (weight, loss_function) in self.losses.items():
                loss_parts[name] = loss_parts[name] + weight * loss_function(
                    pred_t=prediction, target_t=self.state_z[t + 1], mask=self.mask, area=self.area, n_members=self.members
                ) / n_scored

            state = torch.cat([state[:, n_state:], prediction], dim=1)   # feed the prediction back in
            t = t + 1

        # The loss is the sum of the terms; the terms go to the training log (detached: for the log only).
        total_loss = sum(loss_parts.values())
        return total_loss, {name: value.detach() for name, value in loss_parts.items()}

    def training_step(self, start_months):
        # Rollout lengths for this epoch: the schedule entry with the latest first epoch already reached
        # (every schedule starts at epoch 0).
        latest_start = max(first_epoch for first_epoch in self.schedule if first_epoch <= self.current_epoch)
        n_free, n_scored = self.schedule[latest_start]

        loss, loss_parts = self.rollout_loss(start_months, n_free, n_scored)

        # Under mixed precision a NaN loss silently skips the weight update, so the stage would "train"
        # on without changing anything. Stop instead.
        if not torch.isfinite(loss):
            raise FloatingPointError(f"Non-finite training loss in epoch {self.current_epoch}")

        # Write the epoch-mean losses to the training log (metrics.csv).
        self.log("train_loss", loss, prog_bar=True, on_step=False, on_epoch=True, batch_size=len(start_months))
        self.log_dict({f"train_{name}": value for name, value in loss_parts.items()},
                      on_step=False, on_epoch=True, batch_size=len(start_months))
        return loss

    def validation_step(self, start_months):
        # Validation: VALID_STEPS months from the true state, all scored, no free months.
        loss, loss_parts = self.rollout_loss(start_months, 0, VALID_STEPS)
        self.log("val_loss", loss, prog_bar=True, on_step=False, on_epoch=True, batch_size=len(start_months))
        self.log_dict({f"val_{name}": value for name, value in loss_parts.items()},
                      on_step=False, on_epoch=True, batch_size=len(start_months))

    def configure_optimizers(self):
        # Adam, with the learning rate decaying from lr to 0 along a cosine over the stage's epochs.
        optimizer = torch.optim.Adam(self.model.parameters(), lr=self.lr)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=self.epochs)
        return {"optimizer": optimizer, "lr_scheduler": scheduler}

## 8. Train

In [ ]:
# ======================= Stage 1: warm-up, rollouts 1 -> 4 -> 12 months =======================
# If this stage was trained before, its weights are in RUN_DIR/shared/weights.pt: load them and skip training.
stage1_dir = RUN_DIR / "shared"
stage1_weights = stage1_dir / "weights.pt"

if stage1_weights.exists():
    model.load_state_dict(torch.load(stage1_weights, map_location="cpu"))
    print(f"Stage 1: loaded trained weights from {stage1_weights}")

else:
    L.seed_everything(SEED)   # same order of training samples (and same noise) every time

    # What to train, with which losses, and for how long.
    training = RolloutTraining(
        model,
        losses=LOSSES,
        schedule=STAGE1_SCHEDULE,
        lr=STAGE1_LR,
        epochs=STAGE1_EPOCHS,
        members=1,
    )

    # The batches: start months, shuffled for training.
    # A batch is just B start months (time indices): the DataLoader batches the tensor of indices directly.
    train_loader = DataLoader(torch.as_tensor(train_start_months, dtype=torch.long), batch_size=STAGE1_BATCH, shuffle=True)
    valid_loader = DataLoader(torch.as_tensor(valid_start_months, dtype=torch.long), batch_size=STAGE1_BATCH)

    trainer = L.Trainer(
        max_epochs=STAGE1_EPOCHS,
        accelerator="gpu",
        devices=1,
        precision="16-mixed",          # 16-bit arithmetic where it is safe: faster, less memory
        benchmark=True,                # input sizes never change, so let cuDNN pick its fastest kernels once
        num_sanity_val_steps=0,        # skip Lightning's validation dry run before training
        enable_model_summary=False,
        logger=CSVLogger(save_dir=str(stage1_dir), name="", version=""),                 # the log: RUN_DIR/shared/metrics.csv
        callbacks=[ModelCheckpoint(dirpath=stage1_dir / "checkpoints", save_last=True, enable_version_counter=False)],  # always checkpoints/last.ckpt, overwritten every epoch
    )

    # If an earlier run of this stage was cut off (e.g. by a job's walltime), prepare_resume returns its
    # checkpoints/last.ckpt and training carries on from the last finished epoch; otherwise it returns None.
    trainer.fit(training, train_loader, valid_loader, ckpt_path=prepare_resume(stage1_dir))

    torch.save(model.state_dict(), stage1_weights)
    print(f"Stage 1: trained, saved {stage1_weights}")
    torch.cuda.empty_cache()   # give the GPU memory of the finished stage back

In [ ]:
# ======================= Stage 2: pushforward, keep training the same network, scored on months 25-36, then 49-60 =======================
# If this stage was trained before, its weights are in RUN_DIR/deterministic/weights.pt: load them and skip training.
stage2_dir = RUN_DIR / "deterministic"
stage2_weights = stage2_dir / "weights.pt"

if stage2_weights.exists():
    model.load_state_dict(torch.load(stage2_weights, map_location="cpu"))
    print(f"Stage 2: loaded trained weights from {stage2_weights}")

else:
    L.seed_everything(SEED)   # same order of training samples (and same noise) every time

    # What to train, with which losses, and for how long.
    training = RolloutTraining(
        model,
        losses=LOSSES,
        schedule=STAGE2_SCHEDULE,
        lr=STAGE2_LR,
        epochs=STAGE2_EPOCHS,
        members=1,
    )

    # The batches: start months, shuffled for training.
    # A batch is just B start months (time indices): the DataLoader batches the tensor of indices directly.
    train_loader = DataLoader(torch.as_tensor(train_start_months, dtype=torch.long), batch_size=STAGE2_BATCH, shuffle=True)
    valid_loader = DataLoader(torch.as_tensor(valid_start_months, dtype=torch.long), batch_size=STAGE2_BATCH)

    trainer = L.Trainer(
        max_epochs=STAGE2_EPOCHS,
        accelerator="gpu",
        devices=1,
        precision="16-mixed",          # 16-bit arithmetic where it is safe: faster, less memory
        benchmark=True,                # input sizes never change, so let cuDNN pick its fastest kernels once
        num_sanity_val_steps=0,        # skip Lightning's validation dry run before training
        enable_model_summary=False,
        logger=CSVLogger(save_dir=str(stage2_dir), name="", version=""),                 # the log: RUN_DIR/deterministic/metrics.csv
        callbacks=[ModelCheckpoint(dirpath=stage2_dir / "checkpoints", save_last=True, enable_version_counter=False)],  # always checkpoints/last.ckpt, overwritten every epoch
    )

    # If an earlier run of this stage was cut off (e.g. by a job's walltime), prepare_resume returns its
    # checkpoints/last.ckpt and training carries on from the last finished epoch; otherwise it returns None.
    trainer.fit(training, train_loader, valid_loader, ckpt_path=prepare_resume(stage2_dir))

    torch.save(model.state_dict(), stage2_weights)
    print(f"Stage 2: trained, saved {stage2_weights}")
    torch.cuda.empty_cache()   # give the GPU memory of the finished stage back

In [ ]:
# Training curves: the loss per epoch, from each stage's metrics.csv.
# Validation always scores 36 months from the true state, so its loss is comparable across epochs.
fig, axes = plt.subplots(1, 2, figsize=(13, 4), constrained_layout=True)
for ax, stage_folder in zip(axes, ["shared", "deterministic"]):
    metrics = read_metrics(RUN_DIR / stage_folder)   # one row per epoch
    for column in ["train_loss", "val_loss"]:
        rows = metrics.dropna(subset=[column])
        ax.plot(rows["epoch"], rows[column], label=column, lw=2)
    ax.set_title(stage_folder)
    ax.set_xlabel("epoch")
    ax.grid(alpha=0.3)
    ax.legend(frameon=False)
fig.savefig(FIG_DIR / "training_curves.png", dpi=150, bbox_inches="tight")

## 9. Skill test: 2005–2018 with the real forcing

Start from the true state of February 2005 and run freely, with the real forcing, to December 2018
(166 months): one run, never shown the truth again. Results are converted back to physical units as
**anomalies** from the 1970–2000 monthly climatology (z × std).

Baselines: **persistence** (February 2005's anomaly held fixed) and **climatology** (zero anomaly).

The first cell below is the copied rollout code; the second runs it.

In [ ]:
# ==================================================================================================
# IDENTICAL CODE EXISTS IN src/Experiment/evaluation.py
#   autoregressive_rollout
# Copied verbatim so the notebook shows everything it runs. Editing it here does NOT change src/
# (which the test suite checks), and vice versa.
# ==================================================================================================

@torch.no_grad()
def autoregressive_rollout(model, prior, forcing_sequence, mask, target_indices, initial_index, constraint, members=1):
    """
    Free-running rollout from one initial state, for ``members`` members at once (each a separate
    sample for a stochastic model; 1 for a deterministic one).

    prior            (n_prior, P, H, W) the true initial states, oldest first
    forcing_sequence (N, C, H, W)       forcing of each predicted month
    mask             (H, W)             ocean mask
    target_indices   (N,)               time index of each predicted month
    initial_index    int                time index of the last prior state
    constraint       the GlobalBudgetConstraint applied after every step, or None for none
    returns          (members, N, P, H, W) predictions (z-scores), on the CPU
    """
    device = next(model.parameters()).device
    was_training = model.training
    model.eval()
    n_prognostic = prior.shape[1]
    # Every member starts from the same true states, stacked as channels: (members, n_prior * P, H, W).
    state = prior.flatten(0, 1)[None].to(device).repeat(members, 1, 1, 1)
    previous = torch.full((members,), int(initial_index), device=device)   # time index of the current state
    predictions = []
    for k in range(len(forcing_sequence)):
        forcing = forcing_sequence[k : k + 1].to(device).expand(members, -1, -1, -1)
        pred = model(state, forcing, mask.to(device))                        # (members, P, H, W)
        target = torch.full_like(previous, int(target_indices[k]))
        if constraint is not None:
            pred = constraint.to(device)(pred, state[:, -n_prognostic:], previous, target)
        predictions.append(pred.float().cpu())
        state = torch.cat([state[:, n_prognostic:], pred.to(state.dtype)], dim=1)   # drop the oldest month
        previous = target
    model.train(was_training)
    return torch.stack(predictions, dim=1)

In [ ]:
start = month_index[TEST_START]                       # the last true month the model is given
end = month_index[TEST_END] + 1                       # the last month it predicts (2018-12)
target_months = torch.arange(start + 1, end + 1)      # time index of every predicted month
dates = pd.to_datetime([months[i] for i in target_months.tolist()])   # their dates, for the plots

model = model.to(DEVICE)   # the rollout runs wherever the model is

# Free run from the true states of January and February 2005, with the real forcing and the constraint.
z_predicted = autoregressive_rollout(
    model,
    prior=state_z[start - 1 : start + 1],         # (2 months, 2, lat, lon): the true starting states
    forcing_sequence=forcing_z[target_months],    # forcing of every predicted month
    mask=mask,
    target_indices=target_months,                 # the constraint needs every month's index ...
    initial_index=start,                          # ... and the starting month's
    constraint=constraint,
    members=1,
)
# z_predicted: (members, months, 2, lat, lon) z-scores, on the CPU

# Back to physical units, as anomalies from the monthly climatology: anomaly = z x std.
std_of_targets = state_std[target_months]                         # (months, 2, lat, lon)
predicted_anomaly = z_predicted * std_of_targets                  # (members, months, 2, lat, lon)
true_anomaly = state_z[target_months] * std_of_targets            # (months, 2, lat, lon)
persistence_anomaly = state_z[start] * state_std[start]           # (2, lat, lon): February 2005, held fixed

n_members = predicted_anomaly.shape[0]
ocean = mask.numpy() == 1               # (lat, lon) True over the ocean, for the maps
area_weight = area / area.sum()         # (lat, lon) each ocean cell's share of the ocean area; sums to 1

print(f"Predicted {months[target_months[0]]} .. {months[target_months[-1]]}: {len(target_months)} months, {n_members} member(s)")

### Ocean totals

The budget constraint makes the predicted ocean totals follow the surface fluxes, so they track
ACCESS-OM2 by construction (to within how well the monthly data close the budget, about 5% for heat).

In [ ]:
fig, axes = plt.subplots(n_state, 1, figsize=(11, 3.4 * n_state), sharex=True, constrained_layout=True)
for v in range(n_state):   # v = 0: OHC, 1: freshwater content
    ax = axes[v]

    # Ocean total = sum over cells of (anomaly x cell area). Land has area 0.
    member_totals = (predicted_anomaly[:, :, v] * area).sum(dim=(2, 3))   # (members, months)
    member_lines = ax.plot(dates, member_totals.T, color="tab:orange", lw=0.8, alpha=0.6)   # one line per member
    member_lines[0].set_label("emulator")
    true_total = (true_anomaly[:, v] * area).sum(dim=(1, 2))
    ax.plot(dates, true_total, color="k", lw=2, label="ACCESS-OM2")

    ax.set_title(f"Ocean-total {PROGNOSTIC[v]} anomaly")
    ax.set_ylabel(f"{units[v]} x m$^2$")
    ax.grid(alpha=0.3)
    ax.legend(frameon=False)
fig.savefig(FIG_DIR / "skill_ocean_totals.png", dpi=150, bbox_inches="tight")

### Error

RMSE, area-weighted over the ocean, against persistence and climatology. Below climatology = skill.
Once the prediction is no better than climatology, an MSE-trained model tends to fade towards it
(lose variance); the spectral term is there to resist that.

In [ ]:
# The one prediction, (months, 2, lat, lon)
prediction_anomaly = predicted_anomaly[0]

# Area-weighted RMSE: sqrt(sum over cells of area_weight x error^2). Each is (months, 2).
rmse = ((prediction_anomaly - true_anomaly) ** 2 * area_weight).sum(dim=(2, 3)).sqrt()
rmse_persistence = ((persistence_anomaly - true_anomaly) ** 2 * area_weight).sum(dim=(2, 3)).sqrt()
rmse_climatology = (true_anomaly ** 2 * area_weight).sum(dim=(2, 3)).sqrt()   # climatology predicts zero anomaly

fig, axes = plt.subplots(1, n_state, figsize=(14, 4), constrained_layout=True)
for v in range(n_state):
    ax = axes[v]
    ax.plot(dates, rmse[:, v], lw=2, label="emulator")
    ax.plot(dates, rmse_persistence[:, v], color="0.5", lw=1.5, label="persistence")
    ax.plot(dates, rmse_climatology[:, v], color="k", lw=1.5, label="climatology")
    ax.set_title(PROGNOSTIC[v])
    ax.set_ylabel(f"RMSE ({units[v]})")
    ax.grid(alpha=0.3)
    ax.legend(frameon=False, fontsize=8)
fig.savefig(FIG_DIR / "skill_rmse.png", dpi=150, bbox_inches="tight")

# Scores over the first 12 months, the first 36, and the whole test. Ratio < 1 beats climatology.
scores = {}
for v, name in enumerate(PROGNOSTIC):
    scores[name] = {
        "RMSE / climatology, months 1-12": float(rmse[:12, v].mean() / rmse_climatology[:12, v].mean()),
        "RMSE / climatology, months 1-36": float(rmse[:36, v].mean() / rmse_climatology[:36, v].mean()),
        "RMSE / climatology, whole test": float(rmse[:, v].mean() / rmse_climatology[:, v].mean()),
    }
display(pd.DataFrame(scores).T.round(3))

### Maps

Anomaly maps for one period (`"YYYY"` for a year mean, `"YYYY-MM"` for a month): ACCESS-OM2,
the emulator, and the error.

In [ ]:
PERIOD = "2010"   # a year ("2010") or a month ("2010-12")

# Positions of the months in PERIOD: dates whose "YYYY-MM" starts with PERIOD.
in_period = [i for i, date in enumerate(dates) if date.strftime("%Y-%m").startswith(PERIOD)]

titles = ["ACCESS-OM2", "emulator", "emulator - ACCESS-OM2"]
fig, axes = plt.subplots(n_state, 3, figsize=(13.8, 3.2 * n_state), squeeze=False, constrained_layout=True)
for v in range(n_state):
    # Average over the months of PERIOD. Land set to NaN so it shows blank.
    truth_map = true_anomaly[in_period, v].mean(dim=0).numpy()
    emulator_map = predicted_anomaly[0, in_period, v].mean(dim=0).numpy()
    error_map = emulator_map - truth_map
    maps = [np.where(ocean, field, np.nan) for field in [truth_map, emulator_map, error_map]]

    # One colour scale per row: 99th percentile of |truth|.
    limit = np.nanquantile(np.abs(maps[0]), 0.99)
    for column in range(3):
        image = axes[v, column].pcolormesh(longitude, latitude, maps[column], cmap="RdBu_r",
                                           vmin=-limit, vmax=limit, shading="auto")
        axes[v, column].set_title(f"{titles[column]} | {PERIOD}", fontsize=10)
    fig.colorbar(image, ax=axes[v], shrink=0.85, label=f"{PROGNOSTIC[v]} anomaly ({units[v]})")
fig.savefig(FIG_DIR / f"skill_maps_{PERIOD}.png", dpi=150, bbox_inches="tight")

### Zonal spectra

Energy per zonal wavenumber in the first and last year of the test, ACCESS-OM2 against
the emulator (the quantity the spectral loss matches).
Emulator below the truth at long wavelengths = anomalies fading; above it at short wavelengths =
grid-scale noise growing.

The first cell below is the copied spectrum code; the second plots it.

In [ ]:
# ==================================================================================================
# IDENTICAL CODE EXISTS IN src/Experiment/evaluation.py
#   zonal_spectrum
# Copied verbatim so the notebook shows everything it runs. Editing it here does NOT change src/
# (which the test suite checks), and vice versa.
# ==================================================================================================

def zonal_spectrum(anomaly, latitude):
    """
    Time-mean zonal power spectral density of an anomaly (time, latitude, longitude) array, land NaN
    or 0, against physical zonal wavenumber, on the same bands as the spectral loss
    (zonal_wavenumber_bands). Returns (k in cycles/km, PSD in units^2 km). PSD is the area-weighted
    mean zonal variance per unit k, so it reads the same at 1 deg and 1/10 deg.
    """
    rows, band, power_weight, k = zonal_wavenumber_bands(latitude, anomaly.shape[-1])
    x = np.nan_to_num(np.asarray(anomaly, dtype=float))[:, rows, :]
    # Power of every (row, harmonic), time-mean, without the zonal mean; then summed into the bands.
    power = (np.abs(np.fft.rfft(x, axis=-1)) ** 2).mean(0)[:, 1:] * power_weight
    energy = np.bincount(band, power.ravel(), minlength=k.size) / np.cos(np.deg2rad(np.asarray(latitude)[rows])).sum()
    # Energy per band / band width in k = density.
    half = 0.5 / SPECTRAL_BANDS_PER_DECADE
    return k, energy / (k * (10.0**half - 10.0**-half))

In [ ]:
fig, axes = plt.subplots(n_state, 2, figsize=(12, 3.6 * n_state), squeeze=False, constrained_layout=True)
for v in range(n_state):
    for column, (label, year) in enumerate([("first year", slice(0, 12)), ("last year", slice(-12, None))]):
        ax = axes[v, column]

        # zonal_spectrum takes a (months, lat, lon) array; multiplying by ocean (True/False) sets land to 0.
        wavenumber, truth_spectrum = zonal_spectrum(true_anomaly[year, v].numpy() * ocean, latitude)
        wavenumber, emulator_spectrum = zonal_spectrum(predicted_anomaly[0, year, v].numpy() * ocean, latitude)   # emulator

        ax.loglog(wavenumber, truth_spectrum, "-", lw=2, label="ACCESS-OM2")
        ax.loglog(wavenumber, emulator_spectrum, "--", lw=2, label="emulator")
        ax.set_title(f"{PROGNOSTIC[v]}: {label}")
        ax.set_xlabel("zonal wavenumber (cycles/km)")
        ax.set_ylabel("power spectral density")
        ax.grid(alpha=0.3, which="both")
        ax.legend(frameon=False)
fig.savefig(FIG_DIR / "skill_spectra.png", dpi=150, bbox_inches="tight")

## 10. Stability: 50-year control

Repeat the 2003 forcing year 50 times, starting from the true state of January 2003 (one run).
There is no truth to compare with; the question is whether the state stays in the range ACCESS-OM2 visits.

- **Largest |z| over the ocean** each month, on a log scale. ACCESS-OM2's own largest |z| over 2005–2018 is
  the dashed line. Exponential growth shows as a straight rising line.
- **Ocean totals**: fixed by the constraint, so they cannot show drift on their own. That is why the
  |z| panel matters: the constraint fixes only the global total, not local growth.

In `runs/intercomparison_90ep_v3` both UNet emulators passed |z| = 20 around year 12–13 in the 1,000-year
balanced control (`Control_RYF.ipynb`), so watch this panel.

In [ ]:
first = month_index[CONTROL_YEAR[0]]   # January 2003: the starting state
last = month_index[CONTROL_YEAR[1]]    # December 2003

# Predicting month t+1 uses the forcing of month t+1, so one year of steps uses the forcing of
# February 2003 .. January 2004. Repeat that year CONTROL_YEARS times.
one_year = list(range(first + 1, last + 2))
control_months = torch.tensor(one_year * CONTROL_YEARS)   # forcing month (time index) of every step

z_control = autoregressive_rollout(
    model,
    prior=state_z[first - 1 : first + 1],          # true December 2002 and January 2003
    forcing_sequence=forcing_z[control_months],
    mask=mask,
    target_indices=control_months,
    initial_index=first,
    constraint=constraint,
    members=1,
)
z_control = z_control[0]   # the one member: (steps, 2, lat, lon) z-scores

# Largest |z| over the ocean at every step, (steps, 2). Land is set to 0 so it never counts.
land = mask == 0
largest_z = z_control.abs().masked_fill(land, 0).amax(dim=(2, 3))
# ACCESS-OM2's own largest |z| over the skill-test period, (2,), as a reference.
largest_z_truth = state_z[target_months].abs().masked_fill(land, 0).amax(dim=(0, 2, 3))

# Ocean-total anomaly at every step: z x std x area, summed over cells. (steps, 2)
control_total = (z_control * state_std[control_months] * area).sum(dim=(2, 3))

years = np.arange(1, len(control_months) + 1) / 12   # time axis in years

fig, axes = plt.subplots(2, n_state, figsize=(14, 7), sharex=True, constrained_layout=True)
for v in range(n_state):
    axes[0, v].semilogy(years, largest_z[:, v], lw=1.5)
    axes[0, v].axhline(largest_z_truth[v], color="k", ls="--", lw=1, label="ACCESS-OM2 largest, 2005-2018")
    axes[0, v].set_title(f"{PROGNOSTIC[v]}: largest |z| over the ocean")
    axes[0, v].legend(frameon=False, fontsize=8)

    axes[1, v].plot(years, control_total[:, v], lw=1.5)
    axes[1, v].set_title(f"{PROGNOSTIC[v]}: ocean-total anomaly ({units[v]} x m$^2$)")
    axes[1, v].set_xlabel("year of control run")

    axes[0, v].grid(alpha=0.3, which="both")
    axes[1, v].grid(alpha=0.3)
fig.savefig(FIG_DIR / "control_50yr.png", dpi=150, bbox_inches="tight")

# When (if ever) did any cell first pass |z| = 20? The years of every step where one did:
years_above_20 = years[(largest_z > 20).any(dim=1).numpy()]
if len(years_above_20) > 0:
    print(f"largest |z| first above 20 in year {years_above_20[0]:.1f}")
else:
    print("largest |z| never above 20")

## Notes

- Same settings as the `deterministic` emulator of `runs/intercomparison_90ep_v3` (seed 42), written out
  without `ExperimentConfig`. `Control_RYF.ipynb` runs the 1,000-year balanced control.
- The CRPS-ensemble version is `Ocean_Emulator_UNet_CRPS.ipynb`; stage 1 is identical in both.
- Figures are saved to `figures/unet_deterministic/`.
- The copied cells (`IDENTICAL CODE EXISTS IN src/...`) do not follow later changes to `src/`.